# Baseline Technical Analysis — SMA Crossover with ATR Risk

This notebook establishes a **baseline** quantitative workflow for evaluating a technical
strategy. Nothing fancy — the goal is to build a rigorous pipeline we can keep reusing:

1. **Load** every OHLC CSV from `../data/` (filename format `<ASSET>_<TF>_<START>_<END>.csv`).
2. **Describe** a simple, fully-specified strategy (entry, exit, risk plan).
3. **Backtest** it and compute the standard performance metrics (Sharpe, profit factor,
   max drawdown, max consecutive losses, win rate, number of trades, t-statistic on the
   trade distribution, etc.).
4. **Visualize** the results with a dashboard (equity curve, drawdown, PnL distribution,
   rolling win rate).
5. **Optimize** parameters with **Walk-Forward Optimization** (WFO) and inspect the
   stitched out-of-sample performance.
6. **Stress-test** the robustness of the result (Monte-Carlo trade-order shuffling and
   parameter-sensitivity sweeps).

The heavy lifting lives in `../source/` so other notebooks can reuse the same components.


In [ ]:
# Make the `source/` package importable from this notebook's location.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from source import (
    load_all,
    resample_ohlc,
    SMACrossoverStrategy,
    StrategyParams,
    Backtester,
    compute_metrics,
    metrics_table,
    plot_backtest_dashboard,
    plot_wfo_dashboard,
    plot_robustness_dashboard,
    walk_forward,
    monte_carlo_trades,
    monte_carlo_summary,
    parameter_sensitivity,
)

pd.set_option("display.float_format", "{:,.4f}".format)
plt.rcParams["figure.dpi"] = 110


## 1. Load the data

`load_all` scans `../data/`, parses filenames of the form
`<ASSET>_<TIMEFRAME>_<STARTYYYYMMDDHHMM>_<ENDYYYYMMDDHHMM>.csv` and returns a dictionary
keyed by `(asset, timeframe)`. Files whose names don't match the scheme are ignored.


In [ ]:
datasets = load_all(REPO_ROOT / "data")
print(f"Loaded {len(datasets)} dataset(s):")
for (asset, tf), (meta, df) in datasets.items():
    print(f"  {asset} {tf}  rows={len(df):>8,}  {meta.start} -> {meta.end}")


Pick a dataset to work with. We resample minute data to 1-hour bars to keep the
baseline backtest and WFO runtime reasonable — the pipeline is identical on the original
timeframe, only slower.


In [ ]:
if datasets:
    key = next(iter(datasets))
    meta, df_raw = datasets[key]
    print(f"Using {meta.asset} {meta.timeframe}  ({len(df_raw):,} rows)")
    df = resample_ohlc(df_raw, "1h") if meta.timeframe.upper() in {"M1", "M5", "M15", "M30"} else df_raw
    print(f"After resampling: {len(df):,} rows from {df.index[0]} to {df.index[-1]}")
else:
    # Graceful fallback: generate a synthetic random-walk so the notebook still runs
    # end-to-end while the data folder is empty.
    print("No CSV files found in ../data/. Generating a synthetic series so the "
          "notebook can still be executed end-to-end.")
    rng = np.random.default_rng(0)
    idx = pd.date_range("2018-01-01", periods=40000, freq="1h")
    returns = rng.normal(0, 0.0008, size=len(idx))
    close = 1.10 * np.exp(np.cumsum(returns))
    high = close * (1 + rng.uniform(0, 0.0010, size=len(idx)))
    low = close * (1 - rng.uniform(0, 0.0010, size=len(idx)))
    open_ = np.r_[close[0], close[:-1]]
    df = pd.DataFrame({"open": open_, "high": high, "low": low, "close": close}, index=idx)
    meta = None

df.head()


## 2. Strategy definition — SMA crossover with ATR-based risk

**Entry**
- Long when the fast SMA crosses above the slow SMA (bar close).
- Short when the fast SMA crosses below the slow SMA.

**Exit**
- Hard stop loss: `entry - direction * sl_atr_mult * ATR`.
- Take profit: `entry + direction * tp_atr_mult * ATR`.
- Reverse on the opposite crossover.

**Risk plan**
- One unit per trade, PnL measured in price points.
- Intrabar priority: stop loss before take profit (pessimistic assumption).
- No pyramiding, always flat between signals.
- Optional slippage can be passed to the backtester if we want to stress the edge.

This is intentionally plain — the purpose is a **baseline**, not a production system.


In [ ]:
baseline_params = StrategyParams(
    fast=20,
    slow=50,
    atr_period=14,
    sl_atr_mult=2.0,
    tp_atr_mult=3.0,
)
strategy = SMACrossoverStrategy(baseline_params)
backtester = Backtester(strategy, slippage_points=0.0)
result = backtester.run(df)

print(f"Trades: {len(result.trades):,}")
result.trades.head()


### 2.1 Performance metrics

- **num_trades** — sample size; drives statistical power.
- **win_rate / profit_factor / expectancy** — basic edge descriptors.
- **max_drawdown** — largest peak-to-trough equity decline (points).
- **max_consec_losses** — worst losing streak, useful for sizing.
- **sharpe_daily** — Sharpe of daily equity changes, annualized (×√252).
- **sharpe_per_trade** — trade-level Sharpe ≈ mean/std × √n, scale-free alternative.
- **t_stat / p_value** — one-sample t-test against zero expectancy; `p_value` is a
  normal-approximation two-sided p-value, so it's a rough proxy of statistical relevance.


In [ ]:
metrics = compute_metrics(result)
metrics_table(metrics)


### 2.2 Baseline dashboard


In [ ]:
fig = plot_backtest_dashboard(result, title="Baseline SMA crossover")
plt.show()


## 3. Walk-Forward Optimization

We split the series into `n_splits` equal folds. Inside each fold:

- the first `1 - oos_ratio` fraction is **in-sample** (IS): we grid-search the best
  parameters by maximising the daily Sharpe;
- the remaining fraction is **out-of-sample** (OOS): we run the chosen parameters on
  unseen data.

The OOS equity curves are stitched together end-to-end, giving a more honest estimate
of what the strategy would have produced had we re-optimized periodically.


In [ ]:
param_grid = {
    "fast":        [10, 20, 30],
    "slow":        [40, 60, 100],
    "sl_atr_mult": [1.5, 2.0, 3.0],
    "tp_atr_mult": [2.0, 3.0, 4.0],
}

wfo = walk_forward(
    df,
    param_grid=param_grid,
    n_splits=5,
    oos_ratio=0.25,
)
wfo.windows


### 3.1 WFO dashboard

We overlay the stitched OOS equity on the baseline single-run equity so it's easy to
see whether re-optimization is actually helping.


In [ ]:
fig = plot_wfo_dashboard(wfo, full_equity=result.equity)
plt.show()


In [ ]:
# Aggregate WFO OOS metrics for a one-line summary.
if not wfo.oos_trades.empty:
    class _FakeRes:
        pass
    oos_view = _FakeRes()
    oos_view.trades = wfo.oos_trades
    oos_view.equity = wfo.oos_equity
    oos_metrics = compute_metrics(oos_view)
    print("Aggregated out-of-sample metrics:")
    display(metrics_table(oos_metrics))
else:
    print("WFO produced no OOS trades.")


## 4. Robustness

Two complementary tests:

1. **Monte-Carlo trade-order shuffling.** Keeps the trade distribution fixed but
   randomises the order. This reveals how much of the equity-curve shape is luck vs
   edge, and gives a confidence band for final PnL and max drawdown.
2. **Parameter sensitivity.** One-at-a-time sweeps around the baseline params. A robust
   edge should degrade gradually, not fall off a cliff when a parameter moves by one
   notch.


In [ ]:
mc_df = monte_carlo_trades(result.trades, n_runs=1000, seed=42)
summary = monte_carlo_summary(mc_df)
pd.Series(summary).to_frame("value")


In [ ]:
sensitivity = parameter_sensitivity(
    df,
    base_params=baseline_params,
    variations={
        "fast":        [10, 15, 20, 25, 30],
        "slow":        [30, 50, 80, 120],
        "sl_atr_mult": [1.0, 1.5, 2.0, 2.5, 3.0],
        "tp_atr_mult": [1.5, 2.0, 3.0, 4.0, 5.0],
    },
)
sensitivity.head()


In [ ]:
actual_equity = result.trades["pnl_points"].cumsum().values if not result.trades.empty else None
fig = plot_robustness_dashboard(mc_df, baseline_equity=actual_equity, sensitivity=sensitivity)
plt.show()


## 5. Takeaways & next steps

**What this baseline gives us**
- A reproducible load ➜ backtest ➜ metrics ➜ dashboards ➜ WFO ➜ robustness pipeline.
- Reusable classes in `source/` we can plug into other strategies.
- A concrete performance profile to compare future ideas against.

**Known simplifications** (on purpose — these are next iterations, not bugs)
- Execution at bar close, no realistic fill modelling, single unit sizing, no commissions.
- Intra-bar SL-before-TP assumption may under-state TP hits.
- The t-test treats trades as i.i.d., which is only approximately true.
- WFO uses fixed equal folds — anchored/rolling variants with proper retraining cadence
  are a natural follow-up.

**Obvious extensions**
- Volatility-scaled sizing, trailing stops, session/time-of-day filters.
- Classifier/regressor overlays on top of the crossover in the `machine_learning/` folder.
- Portfolio-level backtesting across all datasets discovered by `load_all`.
